# Plastic Pollution Through AI's Eyes — Complete VLM Experiment Pipeline

This notebook implements the paper experiments:

1. **Zero-shot multi-VLM benchmark**
2. **English vs Bangla performance gap**
3. **Image-grounding ablation**: Image+Question vs Question-only
4. **Cross-lingual English–Bangla consistency**
5. **Category-wise performance**
6. **Paired bootstrap confidence intervals**
7. **Human error-analysis template**
8. **Paper-ready tables and figures**

The dataset schema expected by this notebook is the uploaded bilingual JSON with fields such as `id`, `image`, `source_base_id`, `category`, `split`, and six `qa_pairs` per image.

> **Important:** Run the final paper experiment on the **test split only**. The validation split can be used to choose thresholds or debug the evaluation pipeline. Do not tune prompts on the test results.

## 1. Install packages

In [ ]:
# Colab/Jupyter setup
%pip install -q -U openai google-genai transformers accelerate bitsandbytes sentence-transformers scikit-learn pandas numpy matplotlib pillow tqdm tenacity huggingface_hub

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, re, json, math, time, base64, mimetypes, random, gc, platform, sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -------------------------
# Main experiment settings
# -------------------------
TEST_SPLIT = "test"
DEV_MODE = True            # True = small smoke/dev subset; set False for the paper run
DEV_MAX_IMAGES = 10        # only used when DEV_MODE=True
RUN_TEXT_ONLY_ABLATION = True
MAX_NEW_TOKENS = 120
CACHE_EVERY = 1            # save after each prediction: safest for API runs

# Current model IDs are configurable. Keep the exact IDs in the paper.
OPENAI_MODEL = "gpt-5.6-luna"       # swap to gpt-5.6-sol if you want the frontier variant
GEMINI_MODEL = "gemini-3.6-flash"
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
GEMMA_MODEL = "google/gemma-3-4b-it"

# Open-model quantization. 4-bit is practical for Colab GPUs.
USE_4BIT = True

# Cross-lingual threshold is only for a thresholded BCS. Calibrate on validation data if possible.
BCS_THRESHOLD = 0.75

WORKDIR = Path("/content/plastic_vlm_paper") if Path("/content").exists() else Path.cwd() / "plastic_vlm_paper"
WORKDIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = WORKDIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR = WORKDIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)
EXTRACT_DIR = WORKDIR / "dataset_extracted"
EXTRACT_DIR.mkdir(exist_ok=True)

print("WORKDIR:", WORKDIR)
print("Python:", sys.version)
print("Platform:", platform.platform())

## 3. API keys / Hugging Face token

In [ ]:
# Never hard-code keys into a paper notebook that you will publish.
# In Google Colab, add OPENAI_API_KEY, GEMINI_API_KEY and HF_TOKEN under Colab Secrets.

def load_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or load_colab_secret("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or load_colab_secret("GEMINI_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN") or load_colab_secret("HF_TOKEN")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

print("OpenAI key:", "found" if OPENAI_API_KEY else "not set")
print("Gemini key:", "found" if GEMINI_API_KEY else "not set")
print("HF token:", "found" if HF_TOKEN else "not set")

## 4. Locate/upload the dataset

Expected files:
- `Dataset.zip`
- `plastic_waste_vqa_bilingual.json`

The notebook auto-detects names containing `(1)` as well. If auto-detection fails, set `DATASET_ZIP` and `VQA_JSON` manually.

In [ ]:
def first_existing(patterns):
    roots = [Path.cwd(), Path('/content'), Path('/mnt/data')]
    found = []
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            found.extend(root.glob(pat))
    return found[0] if found else None

DATASET_ZIP = first_existing(["Dataset*.zip", "dataset*.zip"])
VQA_JSON = first_existing(["plastic_waste_vqa_bilingual*.json"])

# Optional Colab upload fallback
if (DATASET_ZIP is None or VQA_JSON is None) and Path('/content').exists():
    try:
        from google.colab import files
        print("Upload Dataset.zip and plastic_waste_vqa_bilingual.json")
        uploaded = files.upload()
        DATASET_ZIP = DATASET_ZIP or first_existing(["Dataset*.zip", "dataset*.zip"])
        VQA_JSON = VQA_JSON or first_existing(["plastic_waste_vqa_bilingual*.json"])
    except Exception:
        pass

print("DATASET_ZIP =", DATASET_ZIP)
print("VQA_JSON    =", VQA_JSON)
assert DATASET_ZIP is not None, "Dataset ZIP not found. Set DATASET_ZIP manually."
assert VQA_JSON is not None, "VQA JSON not found. Set VQA_JSON manually."

## 5. Extract images and load the bilingual VQA JSON

In [ ]:
import zipfile

# Extract once
marker = EXTRACT_DIR / '.done'
if not marker.exists():
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    marker.write_text('ok')

with open(VQA_JSON, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print("Image records:", len(dataset))
print("First JSON image path:", dataset[0]['image'])

# The JSON path is typically Image/filename.jpg while ZIP contains Dataset/Image/filename.jpg.
def resolve_image_path(rel_path: str) -> Path:
    rel = Path(rel_path)
    candidates = [
        EXTRACT_DIR / rel,
        EXTRACT_DIR / 'Dataset' / rel,
        EXTRACT_DIR / rel.name,
        EXTRACT_DIR / 'Dataset' / 'Image' / rel.name,
    ]
    for p in candidates:
        if p.exists():
            return p
    # last-resort filename search
    hits = list(EXTRACT_DIR.rglob(rel.name))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not resolve image: {rel_path}")

# Verify every JSON record resolves to an image.
missing = []
for rec in dataset:
    try:
        resolve_image_path(rec['image'])
    except FileNotFoundError:
        missing.append(rec['image'])

print("Missing image paths:", len(missing))
assert not missing, f"Missing images, first examples: {missing[:5]}"

## 6. Flatten QA pairs to a DataFrame and validate the split

In [ ]:
rows = []
for rec in dataset:
    image_path = str(resolve_image_path(rec['image']))
    for qa in rec['qa_pairs']:
        rows.append({
            'image_id': rec['id'],
            'source_base_id': rec.get('source_base_id', rec['id']),
            'image_rel': rec['image'],
            'image_path': image_path,
            'category': rec.get('category', 'uncertain'),
            'visible_categories': rec.get('visible_categories', []),
            'image_quality': rec.get('image_quality'),
            'generation_status': rec.get('generation_status'),
            'quality_flags': rec.get('quality_flags', []),
            'split': rec.get('split'),
            'qa_id': qa['qa_id'],
            'type': qa['type'],
            'difficulty': qa['difficulty'],
            'language': qa['language'],
            'question': qa['question'],
            'reference_answer': qa['answer'],
        })

qa_df = pd.DataFrame(rows)
display(qa_df.head(6))

print("QA rows:", len(qa_df))
print("Images:", qa_df.image_id.nunique())
print("Base images:", qa_df.source_base_id.nunique())
print("\nSplit image counts:")
print(qa_df.groupby('split').image_id.nunique())
print("\nQuestion types:")
print(qa_df['type'].value_counts())
print("\nLanguages:")
print(qa_df['language'].value_counts())
print("\nCategories (image-level):")
print(qa_df.drop_duplicates('image_id')['category'].value_counts())

# Leakage check: a source_base_id must not appear in multiple splits.
base_split_counts = qa_df[['source_base_id','split']].drop_duplicates().groupby('source_base_id')['split'].nunique()
leaked_bases = base_split_counts[base_split_counts > 1]
print("\nSource-base leakage count:", len(leaked_bases))
assert len(leaked_bases) == 0, f"Leakage detected in base IDs: {leaked_bases.index[:10].tolist()}"

# 6 QA pairs per image check
qa_per_image = qa_df.groupby('image_id').size()
print("QA pairs/image distribution:", qa_per_image.value_counts().to_dict())

## 7. Build the test set

For the paper, set `DEV_MODE=False`. In development mode, the notebook samples images rather than individual QA rows, so all six English/Bangla QA pairs for a selected image stay together.

In [ ]:
test_df = qa_df[qa_df['split'].eq(TEST_SPLIT)].copy()

# image-level metadata for optional stratified development sample
image_meta = test_df.drop_duplicates('image_id')[['image_id','category','source_base_id']]

if DEV_MODE:
    # simple deterministic category-aware sample
    selected = []
    per_cat = max(1, math.ceil(DEV_MAX_IMAGES / max(1, image_meta.category.nunique())))
    for cat, g in image_meta.groupby('category', sort=True):
        selected.extend(g.sample(min(per_cat, len(g)), random_state=SEED)['image_id'].tolist())
    selected = selected[:DEV_MAX_IMAGES]
    eval_df = test_df[test_df.image_id.isin(selected)].copy()
else:
    eval_df = test_df.copy()

print("Paper test images available:", test_df.image_id.nunique())
print("Evaluation images selected:", eval_df.image_id.nunique())
print("Evaluation QA rows:", len(eval_df))
print("Potential requests/model with image + text-only:", len(eval_df) * (2 if RUN_TEXT_ONLY_ABLATION else 1))

## 8. Zero-shot prompts

In [ ]:
IMAGE_SYSTEM_INSTRUCTION = """
Answer the visual question directly in the same language as the question.
For visual claims, use only information supported by the image.
For environmental reasoning, use conservative general environmental knowledge and do not invent exact statistics, chemical composition, toxicity, location, brand, or disposal history.
Keep the answer concise: one or two sentences.
Do not provide hidden chain-of-thought; provide only the final answer.
""".strip()

TEXT_ONLY_INSTRUCTION = """
No image is available in this condition. Answer in the same language as the question using only the wording of the question and general knowledge.
Do not invent specific visual details. If the answer cannot be identified without the image, say so briefly.
Keep the answer concise: one or two sentences.
Do not provide hidden chain-of-thought; provide only the final answer.
""".strip()

def build_prompt(question, condition='image'):
    instruction = IMAGE_SYSTEM_INSTRUCTION if condition == 'image' else TEXT_ONLY_INSTRUCTION
    return f"{instruction}\n\nQuestion: {question}\nAnswer:"

## 9. Model adapters — OpenAI, Gemini, Qwen, Gemma

In [ ]:
from tenacity import retry, wait_random_exponential, stop_after_attempt


def image_to_data_url(path):
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or 'image/jpeg'
    b64 = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f"data:{mime};base64,{b64}"

# -------- OpenAI Responses API --------
_openai_client = None

def get_openai_client():
    global _openai_client
    if _openai_client is None:
        if not OPENAI_API_KEY:
            raise RuntimeError("OPENAI_API_KEY is not configured")
        from openai import OpenAI
        _openai_client = OpenAI(api_key=OPENAI_API_KEY)
    return _openai_client

@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(5), reraise=True)
def call_openai(question, image_path=None, condition='image'):
    client = get_openai_client()
    prompt = build_prompt(question, condition)
    content = [{"type": "input_text", "text": prompt}]
    if condition == 'image':
        content.insert(0, {
            "type": "input_image",
            "image_url": image_to_data_url(image_path),
            "detail": "auto",
        })
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[{"role": "user", "content": content}],
        max_output_tokens=MAX_NEW_TOKENS,
    )
    return response.output_text.strip()

# -------- Google Gemini --------
_gemini_client = None

def get_gemini_client():
    global _gemini_client
    if _gemini_client is None:
        if not GEMINI_API_KEY:
            raise RuntimeError("GEMINI_API_KEY is not configured")
        from google import genai
        _gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    return _gemini_client

@retry(wait=wait_random_exponential(min=1, max=30), stop=stop_after_attempt(5), reraise=True)
def call_gemini(question, image_path=None, condition='image'):
    from google.genai import types
    client = get_gemini_client()
    prompt = build_prompt(question, condition)
    contents = [prompt]
    if condition == 'image':
        path = Path(image_path)
        mime = mimetypes.guess_type(path.name)[0] or 'image/jpeg'
        img_part = types.Part.from_bytes(data=path.read_bytes(), mime_type=mime)
        contents = [img_part, prompt]
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=MAX_NEW_TOKENS,
        ),
    )
    return (response.text or '').strip()

# -------- Local Hugging Face VLMs --------
_local_pipe = None
_local_model_id = None

def unload_local_model():
    global _local_pipe, _local_model_id
    _local_pipe = None
    _local_model_id = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def load_local_vlm(model_id):
    global _local_pipe, _local_model_id
    if _local_pipe is not None and _local_model_id == model_id:
        return _local_pipe
    unload_local_model()

    import torch
    from transformers import pipeline, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is strongly recommended for Qwen/Gemma local inference.")

    kwargs = dict(
        task="image-text-to-text",
        model=model_id,
        device_map="auto",
    )
    if USE_4BIT:
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        kwargs['model_kwargs'] = {'quantization_config': quant}
    else:
        kwargs['torch_dtype'] = torch.float16

    print(f"Loading {model_id} ...")
    _local_pipe = pipeline(**kwargs)
    _local_model_id = model_id
    return _local_pipe


def extract_pipeline_text(output):
    if not output:
        return ''
    item = output[0]
    generated = item.get('generated_text', '') if isinstance(item, dict) else item
    # Some chat pipelines return a list of messages.
    if isinstance(generated, list):
        last = generated[-1]
        if isinstance(last, dict):
            content = last.get('content', '')
            if isinstance(content, list):
                texts = [x.get('text','') for x in content if isinstance(x, dict) and x.get('type') == 'text']
                return ' '.join(texts).strip()
            return str(content).strip()
    return str(generated).strip()


def call_local_vlm(model_id, question, image_path=None, condition='image'):
    pipe = load_local_vlm(model_id)
    prompt = build_prompt(question, condition)
    if condition == 'image':
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ]
        }]
        img = Image.open(image_path).convert('RGB')
        out = pipe(
            text=messages,
            images=[img],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            return_full_text=False,
        )
    else:
        messages = [{
            'role': 'user',
            'content': [{'type': 'text', 'text': prompt}]
        }]
        out = pipe(
            text=messages,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            return_full_text=False,
        )
    return extract_pipeline_text(out)


def predict_one(model_key, question, image_path, condition):
    if model_key == 'openai':
        return call_openai(question, image_path, condition)
    if model_key == 'gemini':
        return call_gemini(question, image_path, condition)
    if model_key == 'qwen':
        return call_local_vlm(QWEN_MODEL, question, image_path, condition)
    if model_key == 'gemma':
        return call_local_vlm(GEMMA_MODEL, question, image_path, condition)
    raise ValueError(model_key)

MODEL_LABELS = {
    'openai': OPENAI_MODEL,
    'gemini': GEMINI_MODEL,
    'qwen': QWEN_MODEL,
    'gemma': GEMMA_MODEL,
}

## 10. Smoke test one QA before the expensive run

Run only the models for which you have credentials/hardware. If a model fails here, do **not** start the full experiment.

In [ ]:
smoke = eval_df.iloc[0]
print("Question:", smoke.question)
print("Image:", smoke.image_path)

# Uncomment the model(s) you want to test.
# print("OpenAI:", predict_one('openai', smoke.question, smoke.image_path, 'image'))
# print("Gemini:", predict_one('gemini', smoke.question, smoke.image_path, 'image'))
# print("Qwen:", predict_one('qwen', smoke.question, smoke.image_path, 'image'))
# unload_local_model()
# print("Gemma:", predict_one('gemma', smoke.question, smoke.image_path, 'image'))
# unload_local_model()

## 11. Cached benchmark runner

In [ ]:
def load_cache(cache_path):
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        return df
    return pd.DataFrame()


def run_model(model_key, data=eval_df, conditions=('image','text_only')):
    """Run one model with resumable CSV caching."""
    model_label = MODEL_LABELS[model_key]
    safe = re.sub(r'[^a-zA-Z0-9_.-]+', '_', model_key)
    cache_path = CACHE_DIR / f"predictions_{safe}.csv"
    existing = load_cache(cache_path)
    done = set()
    if not existing.empty:
        done = set(zip(existing['qa_id'], existing['condition']))

    out_rows = [] if existing.empty else existing.to_dict('records')

    active_conditions = list(conditions)
    if not RUN_TEXT_ONLY_ABLATION:
        active_conditions = ['image']

    total_needed = len(data) * len(active_conditions)
    pbar = tqdm(total=total_needed, desc=model_key)
    pbar.update(sum((qa, cond) in done for qa in data.qa_id for cond in active_conditions))

    for row in data.itertuples(index=False):
        for condition in active_conditions:
            key = (row.qa_id, condition)
            if key in done:
                continue
            t0 = time.time()
            error = ''
            pred = ''
            try:
                pred = predict_one(model_key, row.question, row.image_path, condition)
            except Exception as e:
                error = repr(e)
                print(f"\nERROR {model_key} {row.qa_id} {condition}: {error}")
            out_rows.append({
                'model_key': model_key,
                'model': model_label,
                'condition': condition,
                'image_id': row.image_id,
                'source_base_id': row.source_base_id,
                'image_path': row.image_path,
                'category': row.category,
                'split': row.split,
                'qa_id': row.qa_id,
                'type': row.type,
                'difficulty': row.difficulty,
                'language': row.language,
                'question': row.question,
                'reference_answer': row.reference_answer,
                'prediction': pred,
                'latency_sec': time.time() - t0,
                'error': error,
            })
            done.add(key)
            if len(out_rows) % CACHE_EVERY == 0:
                pd.DataFrame(out_rows).to_csv(cache_path, index=False, encoding='utf-8-sig')
            pbar.update(1)

    pbar.close()
    result = pd.DataFrame(out_rows)
    result.to_csv(cache_path, index=False, encoding='utf-8-sig')
    return result

# Recommended execution order:
# 1) cloud models (no GPU memory issue)
# openai_pred = run_model('openai')
# gemini_pred = run_model('gemini')
#
# 2) one local model at a time
# qwen_pred = run_model('qwen')
# unload_local_model()
# gemma_pred = run_model('gemma')
# unload_local_model()

## 12. Load all completed model caches

This lets you close/restart the notebook and continue analysis without re-querying APIs.

In [ ]:
def collect_prediction_caches():
    dfs = []
    for p in sorted(CACHE_DIR.glob('predictions_*.csv')):
        d = pd.read_csv(p)
        if len(d):
            dfs.append(d)
            print(p.name, len(d))
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

pred_df = collect_prediction_caches()
print("Total cached predictions:", len(pred_df))
if len(pred_df):
    display(pred_df.head())

## 13. Recognition category scoring

In [ ]:
CATEGORY_PATTERNS = {
    'plastic_bottle': [
        r'\bplastic bottle\b', r'\bbottle\b', r'প্লাস্টিক(?:ের)? বোতল', r'বোতল'
    ],
    'plastic_cup': [
        r'\bplastic cup\b', r'\bcup[- ]?like\b', r'\bcup\b', r'প্লাস্টিক কাপ', r'কাপসদৃশ', r'কাপ'
    ],
    'plastic_bag': [
        r'\bplastic bag\b', r'\bpolythene\b', r'\bbag\b', r'প্লাস্টিক ব্যাগ', r'পলিথিন', r'ব্যাগ'
    ],
    'plastic_straw': [
        r'\bplastic straw\b', r'\bstraw\b', r'প্লাস্টিক স্ট্র', r'স্ট্র'
    ],
    'plastic_wrapper': [
        r'\bplastic wrapper\b', r'\bwrapper\b', r'\bflexible plastic packaging\b', r'\bplastic packaging\b',
        r'র.?্যাপার', r'প্লাস্টিক প্যাকেজিং', r'নমনীয় প্লাস্টিক প্যাকেজিং'
    ],
}


def categories_in_text(text):
    text = str(text).lower()
    found = set()
    for cat, pats in CATEGORY_PATTERNS.items():
        if any(re.search(p, text, flags=re.I) for p in pats):
            found.add(cat)
    return found


def strict_recognition_score(pred, true_cat):
    cats = categories_in_text(pred)
    return float(cats == {true_cat})


def relaxed_recognition_score(pred, true_cat):
    return float(true_cat in categories_in_text(pred))

# quick sanity examples
for x in ["plastic bottle", "প্লাস্টিকের বোতল", "flexible plastic packaging", "প্লাস্টিক কাপ"]:
    print(x, '->', categories_in_text(x))

## 14. Multilingual semantic similarity for open-ended answers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import paired_cosine_distances

EMBED_MODEL_ID = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
embedder = SentenceTransformer(EMBED_MODEL_ID)


def cosine_sim_pairs(a, b, batch_size=64):
    a = [str(x) for x in a]
    b = [str(x) for x in b]
    ea = embedder.encode(a, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
    eb = embedder.encode(b, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
    return np.sum(ea * eb, axis=1)


def add_automatic_scores(df):
    df = df.copy()
    ok = df['error'].fillna('').eq('') & df['prediction'].fillna('').str.len().gt(0)
    df['semantic_similarity'] = np.nan
    if ok.any():
        df.loc[ok, 'semantic_similarity'] = cosine_sim_pairs(
            df.loc[ok, 'prediction'].tolist(),
            df.loc[ok, 'reference_answer'].tolist(),
        )

    df['recognition_strict'] = np.nan
    df['recognition_relaxed'] = np.nan
    rec = ok & df['type'].eq('recognition')
    df.loc[rec, 'recognition_strict'] = [
        strict_recognition_score(p, c) for p, c in zip(df.loc[rec,'prediction'], df.loc[rec,'category'])
    ]
    df.loc[rec, 'recognition_relaxed'] = [
        relaxed_recognition_score(p, c) for p, c in zip(df.loc[rec,'prediction'], df.loc[rec,'category'])
    ]

    # task_score combines recognition accuracy with semantic similarity for the two open-ended tasks.
    # Report per-task results in the paper; do not over-interpret a mixed overall average.
    df['task_score'] = df['semantic_similarity']
    df.loc[df['type'].eq('recognition'), 'task_score'] = df.loc[df['type'].eq('recognition'), 'recognition_strict']
    return df

if len(pred_df):
    scored_df = add_automatic_scores(pred_df)
    scored_df.to_csv(RESULTS_DIR/'all_scored_predictions.csv', index=False, encoding='utf-8-sig')
    display(scored_df[['model','condition','type','language','prediction','task_score']].head())
else:
    scored_df = pd.DataFrame()

## 15. Experiment 1 — Main zero-shot benchmark

In [ ]:
def main_benchmark_table(scored):
    img = scored[scored.condition.eq('image')].copy()
    # Per task: recognition uses accuracy; description/reasoning use semantic similarity.
    table = img.groupby(['model','type','language'], as_index=False)['task_score'].agg(['mean','count','std']).reset_index()
    table['se'] = table['std'] / np.sqrt(table['count'].clip(lower=1))
    table['ci95_low'] = table['mean'] - 1.96*table['se']
    table['ci95_high'] = table['mean'] + 1.96*table['se']
    return table

if len(scored_df):
    main_table = main_benchmark_table(scored_df)
    display(main_table)
    main_table.to_csv(RESULTS_DIR/'table_main_benchmark.csv', index=False)

## 16. Experiment 2 — English vs Bangla performance gap

In [ ]:
def language_gap_table(scored):
    img = scored[scored.condition.eq('image')].copy()
    means = img.groupby(['model','type','language'])['task_score'].mean().unstack('language')
    for col in ['English','Bangla']:
        if col not in means.columns:
            means[col] = np.nan
    means['EN_minus_BN'] = means['English'] - means['Bangla']
    return means.reset_index()

if len(scored_df):
    lang_gap = language_gap_table(scored_df)
    display(lang_gap)
    lang_gap.to_csv(RESULTS_DIR/'table_language_gap.csv', index=False)

    plot_df = lang_gap.copy()
    labels = plot_df['model'].astype(str) + ' | ' + plot_df['type'].astype(str)
    plt.figure(figsize=(11,5))
    plt.bar(labels, plot_df['EN_minus_BN'])
    plt.axhline(0, linewidth=1)
    plt.ylabel('English score − Bangla score')
    plt.title('English–Bangla Performance Gap')
    plt.xticks(rotation=70, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/'fig_language_gap.png', dpi=220, bbox_inches='tight')
    plt.show()

## 17. Experiment 3 — Image-grounding ablation

In [ ]:
def grounding_gain_table(scored):
    x = scored.pivot_table(
        index=['model','qa_id','image_id','type','language','category'],
        columns='condition',
        values='task_score',
        aggfunc='first'
    ).reset_index()
    if 'image' not in x or 'text_only' not in x:
        raise ValueError('Both image and text_only conditions are required.')
    x['visual_grounding_gain'] = x['image'] - x['text_only']
    summary = x.groupby(['model','type','language'], as_index=False).agg(
        image_score=('image','mean'),
        text_only_score=('text_only','mean'),
        visual_grounding_gain=('visual_grounding_gain','mean'),
        n=('visual_grounding_gain','count'),
    )
    return x, summary

if len(scored_df) and {'image','text_only'}.issubset(set(scored_df.condition.dropna().unique())):
    grounding_pairs, grounding_summary = grounding_gain_table(scored_df)
    display(grounding_summary)
    grounding_pairs.to_csv(RESULTS_DIR/'grounding_paired_scores.csv', index=False)
    grounding_summary.to_csv(RESULTS_DIR/'table_grounding_gain.csv', index=False)

    labels = grounding_summary['model'].astype(str) + ' | ' + grounding_summary['type'].astype(str) + ' | ' + grounding_summary['language'].astype(str)
    plt.figure(figsize=(12,5))
    plt.bar(labels, grounding_summary['visual_grounding_gain'])
    plt.axhline(0, linewidth=1)
    plt.ylabel('Image score − Question-only score')
    plt.title('Visual Grounding Gain')
    plt.xticks(rotation=75, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/'fig_grounding_gain.png', dpi=220, bbox_inches='tight')
    plt.show()

## 18. Experiment 4 — Cross-lingual consistency

In [ ]:
def cross_lingual_consistency(scored, condition='image', threshold=BCS_THRESHOLD):
    d = scored[scored.condition.eq(condition)].copy()
    en = d[d.language.eq('English')].copy()
    bn = d[d.language.eq('Bangla')].copy()

    # Each image has one EN and one BN question for each task type.
    paired = en.merge(
        bn,
        on=['model','image_id','source_base_id','type','category','condition'],
        suffixes=('_en','_bn')
    )
    paired['crosslingual_similarity'] = cosine_sim_pairs(
        paired['prediction_en'].fillna('').tolist(),
        paired['prediction_bn'].fillna('').tolist(),
    )
    paired['bcs_consistent'] = (paired['crosslingual_similarity'] >= threshold).astype(int)

    # Cleaner categorical consistency for recognition.
    def cat_set_equal(a, b):
        ca, cb = categories_in_text(a), categories_in_text(b)
        return float(len(ca) > 0 and ca == cb)

    paired['recognition_category_consistency'] = np.nan
    m = paired['type'].eq('recognition')
    paired.loc[m, 'recognition_category_consistency'] = [
        cat_set_equal(a,b) for a,b in zip(paired.loc[m,'prediction_en'], paired.loc[m,'prediction_bn'])
    ]

    summary = paired.groupby(['model','type'], as_index=False).agg(
        mean_crosslingual_similarity=('crosslingual_similarity','mean'),
        thresholded_BCS=('bcs_consistent','mean'),
        recognition_category_consistency=('recognition_category_consistency','mean'),
        n=('crosslingual_similarity','count')
    )
    return paired, summary

if len(scored_df):
    cl_pairs, cl_summary = cross_lingual_consistency(scored_df, 'image')
    display(cl_summary)
    cl_pairs.to_csv(RESULTS_DIR/'cross_lingual_pairs.csv', index=False, encoding='utf-8-sig')
    cl_summary.to_csv(RESULTS_DIR/'table_cross_lingual_consistency.csv', index=False)

    labels = cl_summary['model'].astype(str) + ' | ' + cl_summary['type'].astype(str)
    plt.figure(figsize=(10,5))
    plt.bar(labels, cl_summary['mean_crosslingual_similarity'])
    plt.ylim(0,1)
    plt.ylabel('Mean EN–BN output cosine similarity')
    plt.title('Cross-Lingual Output Consistency')
    plt.xticks(rotation=70, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR/'fig_cross_lingual_consistency.png', dpi=220, bbox_inches='tight')
    plt.show()

## 19. Category-wise recognition performance (macro vs micro)

In [ ]:
def category_recognition_table(scored):
    d = scored[(scored.condition.eq('image')) & (scored.type.eq('recognition'))].copy()
    table = d.groupby(['model','language','category'], as_index=False).agg(
        strict_accuracy=('recognition_strict','mean'),
        relaxed_accuracy=('recognition_relaxed','mean'),
        n=('qa_id','count')
    )
    macro = table.groupby(['model','language'], as_index=False)['strict_accuracy'].mean().rename(columns={'strict_accuracy':'macro_category_accuracy'})
    micro = d.groupby(['model','language'], as_index=False)['recognition_strict'].mean().rename(columns={'recognition_strict':'micro_accuracy'})
    return table, macro.merge(micro, on=['model','language'])

if len(scored_df):
    cat_table, macro_micro = category_recognition_table(scored_df)
    display(cat_table)
    display(macro_micro)
    cat_table.to_csv(RESULTS_DIR/'table_category_recognition.csv', index=False)
    macro_micro.to_csv(RESULTS_DIR/'table_macro_micro_recognition.csv', index=False)

## 20. Paired bootstrap 95% confidence intervals

In [ ]:
def paired_bootstrap_difference(a, b, n_boot=5000, seed=SEED):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) == 0:
        return {'n':0, 'mean_diff':np.nan, 'ci95_low':np.nan, 'ci95_high':np.nan}
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    n = len(a)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = np.mean(a[idx] - b[idx])
    return {
        'n': n,
        'mean_diff': float(np.mean(a-b)),
        'ci95_low': float(np.percentile(diffs, 2.5)),
        'ci95_high': float(np.percentile(diffs, 97.5)),
    }


def language_gap_bootstrap(scored):
    d = scored[scored.condition.eq('image')].copy()
    en = d[d.language.eq('English')]
    bn = d[d.language.eq('Bangla')]
    pairs = en.merge(bn, on=['model','image_id','type','category','condition'], suffixes=('_en','_bn'))
    rows=[]
    for (model, typ), g in pairs.groupby(['model','type']):
        r=paired_bootstrap_difference(g.task_score_en, g.task_score_bn)
        rows.append({'model':model,'type':typ,**r})
    return pd.DataFrame(rows)


def grounding_bootstrap(scored):
    x = scored.pivot_table(index=['model','qa_id','type','language'], columns='condition', values='task_score', aggfunc='first').reset_index()
    rows=[]
    if not {'image','text_only'}.issubset(x.columns):
        return pd.DataFrame()
    for (model, typ, lang), g in x.groupby(['model','type','language']):
        r=paired_bootstrap_difference(g['image'], g['text_only'])
        rows.append({'model':model,'type':typ,'language':lang,**r})
    return pd.DataFrame(rows)

if len(scored_df):
    lg_ci = language_gap_bootstrap(scored_df)
    display(lg_ci)
    lg_ci.to_csv(RESULTS_DIR/'table_language_gap_bootstrap_ci.csv', index=False)
    gg_ci = grounding_bootstrap(scored_df)
    if len(gg_ci):
        display(gg_ci)
        gg_ci.to_csv(RESULTS_DIR/'table_grounding_bootstrap_ci.csv', index=False)

## 21. Human error analysis

Do not pretend automated semantic similarity can fully measure factual grounding or hallucination. For the paper, manually review a stratified subset.

Suggested labels:
- `object_misclassification`
- `visual_hallucination`
- `environmental_hallucination`
- `bangla_misunderstanding`
- `generic_environmental_answer`
- `other`
- `none`

Score each response:
- `correctness_0_2`
- `relevance_0_2`
- `grounding_0_2`
- `hallucination_0_1`

In [ ]:
ERROR_TYPES = [
    'none',
    'object_misclassification',
    'visual_hallucination',
    'environmental_hallucination',
    'bangla_misunderstanding',
    'generic_environmental_answer',
    'other',
]


def make_human_annotation_template(scored, n=120, seed=SEED):
    d = scored[scored.condition.eq('image')].copy()
    # Balanced-ish by model/type/language. Sampling with group fraction then cap.
    groups = list(d.groupby(['model','type','language']))
    per_group = max(1, math.ceil(n / max(1, len(groups))))
    parts=[]
    for _, g in groups:
        parts.append(g.sample(min(per_group, len(g)), random_state=seed))
    sample = pd.concat(parts, ignore_index=True).drop_duplicates(['model','qa_id']).head(n).copy()
    keep = [
        'model','image_id','image_path','category','qa_id','type','language',
        'question','reference_answer','prediction'
    ]
    sample = sample[keep]
    sample['annotator'] = ''
    sample['correctness_0_2'] = ''
    sample['relevance_0_2'] = ''
    sample['grounding_0_2'] = ''
    sample['hallucination_0_1'] = ''
    sample['error_type'] = ''
    sample['notes'] = ''
    return sample

if len(scored_df):
    human_template = make_human_annotation_template(scored_df, n=min(120, len(scored_df)))
    human_path = RESULTS_DIR/'human_error_annotation_template.csv'
    human_template.to_csv(human_path, index=False, encoding='utf-8-sig')
    print("Saved:", human_path)
    display(human_template.head())

## 22. Optional annotation viewer

In [ ]:
def show_annotation_item(template_df, idx=0):
    row = template_df.iloc[idx]
    print(f"[{idx+1}/{len(template_df)}] {row['model']} | {row['type']} | {row['language']} | {row['category']}")
    print("Question:", row['question'])
    print("Reference:", row['reference_answer'])
    print("Prediction:", row['prediction'])
    display(Image.open(row['image_path']).convert('RGB'))

# Example:
# show_annotation_item(human_template, 0)

## 23. Aggregate completed human annotations

In [ ]:
def aggregate_human_annotations(csv_path):
    h = pd.read_csv(csv_path)
    for c in ['correctness_0_2','relevance_0_2','grounding_0_2','hallucination_0_1']:
        h[c] = pd.to_numeric(h[c], errors='coerce')
    h['human_total_0_6'] = h[['correctness_0_2','relevance_0_2','grounding_0_2']].sum(axis=1, min_count=3)
    summary = h.groupby(['model','type','language'], as_index=False).agg(
        correctness=('correctness_0_2','mean'),
        relevance=('relevance_0_2','mean'),
        grounding=('grounding_0_2','mean'),
        hallucination_rate=('hallucination_0_1','mean'),
        human_total_0_6=('human_total_0_6','mean'),
        n=('qa_id','count'),
    )
    errors = h[h.error_type.notna() & h.error_type.ne('')].groupby(['model','error_type']).size().reset_index(name='count')
    return h, summary, errors

# After annotation:
# annotated, human_summary, error_counts = aggregate_human_annotations(RESULTS_DIR/'human_error_annotation_completed.csv')
# display(human_summary)
# display(error_counts)
# human_summary.to_csv(RESULTS_DIR/'table_human_evaluation.csv', index=False)
# error_counts.to_csv(RESULTS_DIR/'table_error_counts.csv', index=False)

## 24. Optional two-annotator agreement

In [ ]:
from sklearn.metrics import cohen_kappa_score

def annotator_agreement(csv_a, csv_b):
    a = pd.read_csv(csv_a)
    b = pd.read_csv(csv_b)
    m = a.merge(b, on=['model','qa_id'], suffixes=('_a','_b'))
    results = {}
    for c in ['correctness_0_2','relevance_0_2','grounding_0_2','hallucination_0_1','error_type']:
        ca, cb = f'{c}_a', f'{c}_b'
        valid = m[ca].notna() & m[cb].notna()
        if valid.any():
            results[c] = cohen_kappa_score(m.loc[valid,ca], m.loc[valid,cb])
    return results

# Example:
# annotator_agreement('annotator_A.csv', 'annotator_B.csv')

## 25. Failure examples for the qualitative section

In [ ]:
def lowest_scoring_examples(scored, model=None, typ=None, language=None, n=10):
    d = scored[scored.condition.eq('image')].copy()
    if model is not None: d = d[d.model.eq(model)]
    if typ is not None: d = d[d.type.eq(typ)]
    if language is not None: d = d[d.language.eq(language)]
    return d.sort_values('task_score').head(n)[[
        'model','image_path','category','type','language','question','reference_answer','prediction','task_score'
    ]]

# Example:
# display(lowest_scoring_examples(scored_df, typ='environmental_reasoning', language='Bangla', n=10))

## 26. Save experiment metadata for reproducibility

In [ ]:
import importlib.metadata as im
from datetime import datetime, timezone

packages = ['openai','google-genai','transformers','accelerate','bitsandbytes','sentence-transformers','scikit-learn','pandas','numpy','Pillow']
versions = {}
for p in packages:
    try:
        versions[p] = im.version(p)
    except Exception:
        versions[p] = None

metadata = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'seed': SEED,
    'dev_mode': DEV_MODE,
    'test_split': TEST_SPLIT,
    'model_ids': MODEL_LABELS,
    'embedding_model': EMBED_MODEL_ID,
    'bcs_threshold': BCS_THRESHOLD,
    'max_new_tokens': MAX_NEW_TOKENS,
    'python': sys.version,
    'packages': versions,
    'dataset_json': str(VQA_JSON),
    'dataset_zip': str(DATASET_ZIP),
    'test_images': int(test_df.image_id.nunique()),
    'test_qa_pairs': int(len(test_df)),
}
with open(RESULTS_DIR/'experiment_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 27. Final paper-ready result manifest

In [ ]:
print("RESULTS DIRECTORY:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob('*')):
    print('-', p.name)

print("\nFor the final paper run:")
print("1. Set DEV_MODE=False and restart from the test-set cell.")
print("2. Run the same zero-shot prompt for every model.")
print("3. Do not change prompts after looking at test scores.")
print("4. Report per-task EN/BN scores, grounding gain, consistency, category macro/micro, and human error analysis.")